# Sprint 0 - Data Acquisition & Exploration

**Project:** Smart Crop Disease Detection and Assistant (see `AGENT.md` / `final_brief_and_plan.md`)

**Goal of this notebook (Sprint 0 "done when"):** download both datasets, organize them into
`train/val/test` folders by class, and load a labeled batch of images.

| Dataset | Source | Images | Classes |
|---|---|---|---|
| PlantVillage | GitHub `spMohanty/PlantVillage-Dataset` (raw/color) | 54,305 | 38 |
| PlantDoc | GitHub `pratikkayal/PlantDoc-Dataset` | 2,578 | 28 |

**How to use:** `Runtime -> Run all`. The **raw** data is saved to Google Drive so it survives
session disconnects; the organized `train/val/test` splits are built **locally** in `/content`
(they are deterministic from the raw data + seed 42, so they don't need to live on Drive).

**Pipeline:**
1. Mount Drive + clone this repo
2. Install dependencies
3. *(optional)* Reset `folium/data` on Drive - do this once after the partial-download incident
4. `scripts/download_datasets.py` -> `data/plantvillage/raw`, `data/plantdoc/raw` + `manifest.json`
5. `scripts/organize_datasets.py` -> local `/content/folium_data/{train,val,test}/<class>` + `class_map.json`
6. Load a batch of images with `torchvision.ImageFolder` + `DataLoader` and inspect it
7. **Durability gate** - verify Drive's raw data from a *fresh* session (see Step 7)

In [ ]:
import platform
import subprocess
import sys

print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
try:
    gpu = subprocess.run(["nvidia-smi"], capture_output=True, text=True, timeout=30)
    print(gpu.stdout.strip() or gpu.stderr.strip() or "No GPU detected (CPU only)")
except Exception as exc:
    print("GPU check skipped:", exc)

## Step 1 - Mount Google Drive and clone the repo

- **Drive** holds `folium/data` so the ~1 GB of raw images persists across Colab sessions.
- **Repo** is cloned fresh (or reused) so the latest scripts under `scripts/` are available.
- **Local** (`/content/folium_data`) is where organized splits are written each session.

In [ ]:
from google.colab import drive

# Mount Google Drive so the raw data persists across sessions.
# A popup asks you to authorize - click through and allow access.
drive.mount("/content/drive")

from pathlib import Path

REPO_URL = "https://github.com/io-PEAK/folium.git"   # change if you forked
REPO_DIR = Path("/content/folium")
DATA_DIR = Path("/content/drive/MyDrive/folium/data")  # durable raw data
LOCAL_DATA_DIR = Path("/content/folium_data")          # per-session organized splits

if not (REPO_DIR / "scripts").exists():
    %cd /content
    !git clone --depth 1 {REPO_URL}
else:
    print(f"repo already cloned at {REPO_DIR}")

DATA_DIR.mkdir(parents=True, exist_ok=True)
LOCAL_DATA_DIR.mkdir(parents=True, exist_ok=True)
print("DATA_DIR:", DATA_DIR)
print("LOCAL_DATA_DIR:", LOCAL_DATA_DIR)

## Step 2 - Install dependencies

Training deps (`torch`, `torchvision`, `albumentations`) are installed now so we
don't have to wait again in Sprint 1.

In [ ]:
%pip install -q --upgrade pip
%pip install -q torch torchvision albumentations matplotlib pandas tqdm opencv-python-headless

## Step 3 - (Once) Reset the Drive data folder

On 11 Aug the first download ended with Drive holding only a **partial** dataset: the session's
FUSE cache made reads look complete while Drive's backend never received the tail of the writes
(the last PlantVillage classes, all of PlantDoc, and `class_map.json` were lost).

Set `RESET_DATA = True` **one time** to delete `folium/data` and re-download from scratch, then
flip it back to `False`. Never re-run this after the data is verified durable.

In [ ]:
RESET_DATA = True   # set True exactly once to wipe folium/data, then back to False

if RESET_DATA:
    !rm -rf {DATA_DIR}
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    print(f"Reset: deleted and recreated {DATA_DIR}")
else:
    print("RESET_DATA=False; keeping existing Drive data")

## Step 4 - Download the datasets

Runs `scripts/download_datasets.py`:
- **PlantDoc first** (small, fast), cloned in `--work-dir` then moved into Drive
- **PlantVillage**: sparse checkout of `raw/color` (~1 GB) into `--work-dir`, moved into Drive
- Writes `<data-dir>/<name>/manifest.json` with per-class counts measured from the checkout
- **Resumable**: on re-run, only classes whose on-disk count differs from expected are fetched

If a dataset is already verified it is skipped (use `--force` to re-download).

In [ ]:
import subprocess
import sys

WORK_DIR = Path("/content/folium_cache")  # session-only scratch (clones/checkouts)
WORK_DIR.mkdir(parents=True, exist_ok=True)

cmd = [
    sys.executable,
    str(REPO_DIR / "scripts" / "download_datasets.py"),
    "--data-dir", str(DATA_DIR),
    "--work-dir", str(WORK_DIR),
    "--dataset", "all",
]
result = subprocess.run(cmd, cwd=str(REPO_DIR))
assert result.returncode == 0, "download_datasets.py failed"

## Step 5 - Organize into train/val/test (local)

Reads raw from Drive (`--raw-dir`) and writes the splits **locally** (`--data-dir`):
- **PlantVillage**: stratified 80/10/10 split (seed 42)
- **PlantDoc**: keeps its shipped test set, carves 10% of training images as validation
- Writes `class_map.json` (PlantDoc -> PlantVillage class mapping)

Local splits avoid ~800 MB of risky Drive FUSE writes each session; they are rebuilt
deterministically from the durable raw data.

In [ ]:
result = subprocess.run([
    sys.executable,
    str(REPO_DIR / "scripts" / "organize_datasets.py"),
    "--raw-dir", str(DATA_DIR),
    "--data-dir", str(LOCAL_DATA_DIR),
], cwd=str(REPO_DIR))
assert result.returncode == 0, "organize_datasets.py failed"

## Step 6 - Verify: load a labeled batch

The "done" check for Sprint 0: load images + labels with `torchvision.ImageFolder`
and pull one batch from a `DataLoader`.

In [ ]:
import torch
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

transform = transforms.Compose([transforms.Resize((256, 256)), transforms.ToTensor()])
pv_train = datasets.ImageFolder(LOCAL_DATA_DIR / "plantvillage" / "train", transform=transform)
loader = DataLoader(pv_train, batch_size=32, shuffle=True, num_workers=2)

images, labels = next(iter(loader))
print("One batch: images", tuple(images.shape), "labels", tuple(labels.shape))
print("dtype/range: images", images.dtype, round(float(images.min()), 2), "-", round(float(images.max()), 2))
print("num train classes:", len(pv_train.classes))
print("num train images:", len(pv_train))
assert len(pv_train) > 0 and len(pv_train.classes) == 38
print("SPRINT 0 DONE: can load a labeled batch of images")

## Step 7 - Durability gate: verify Drive's RAW data

A download is only trustworthy once a **fresh** Colab session (no local FUSE cache from the
session that wrote it) confirms the per-class counts on Drive match the expected counts.

Run this cell now, then **open a brand-new session**, run Step 1 + this cell again, and confirm
it still says PASS there too.

> The expected tables come from `scripts/download_datasets.py`, measured from the upstream
> git trees with `git ls-tree` (so PlantDoc's counts include the 6 case-duplicate files that
> only a Linux/Colab clone keeps).

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(REPO_DIR))

from scripts.download_datasets import PLANTVILLAGE_EXPECTED, PLANTDOC_EXPECTED, _count_images


def check(root: Path, expected: dict) -> list:
    """Return list of (class, got, want) mismatches."""
    bad = []
    for cls, want in sorted(expected.items()):
        got = _count_images(root / cls) if (root / cls).exists() else 0
        if got != want:
            bad.append((cls, got, want))
    return bad


pv_bad = check(DATA_DIR / "plantvillage" / "raw", PLANTVILLAGE_EXPECTED)
pd_train_bad = check(DATA_DIR / "plantdoc" / "raw" / "train", PLANTDOC_EXPECTED["train"])
pd_test_bad = check(DATA_DIR / "plantdoc" / "raw" / "test", PLANTDOC_EXPECTED["test"])


def report(name: str, bad: list) -> None:
    print(f"{name}: {'OK' if not bad else 'MISMATCH (' + str(len(bad)) + ' classes)'}")
    for cls, got, want in bad[:10]:
        print(f"    {cls}: {got} != {want}")
    if len(bad) > 10:
        print(f"    ... and {len(bad) - 10} more")


report("plantvillage/raw (expect 38 classes)", pv_bad)
report("plantdoc/raw/train (expect 28 classes)", pd_train_bad)
report("plantdoc/raw/test (expect 27 classes)", pd_test_bad)

pv_total = sum(_count_images(d) for d in (DATA_DIR / "plantvillage" / "raw").iterdir() if d.is_dir())
print("\nTOTAL plantvillage images on Drive:", pv_total)

if not (pv_bad or pd_train_bad or pd_test_bad):
    print("\nDURABILITY GATE: PASS in this session")
    print("NOW open a NEW Colab session and re-run Step 1 + this cell.")
    print("If it PASSES there too, the raw data is durably on Drive.")
    print("If it FAILS there, re-run Step 4 (download) in that session.")
else:
    print("\nDURABILITY GATE: FAIL - re-run Step 4 (download) before trusting the data.")

### Sample images from the batch (resized to 256x256)

In [ ]:
import matplotlib.pyplot as plt
import torchvision.utils as vutils

grid = vutils.make_grid(images[:16], nrow=4, normalize=True).permute(1, 2, 0).numpy()
plt.figure(figsize=(12, 12))
plt.imshow(grid)
plt.axis("off")
plt.title("First 16 images (resized 256x256)")
plt.show()

for i in range(8):
    print(f"  {i}: {pv_train.classes[labels[i].item()]}")

### Train class distribution (PlantVillage)

In [ ]:
import collections

counts = collections.Counter(pv_train.targets)
ordered = sorted(counts.items(), key=lambda kv: kv[1], reverse=True)
names = [pv_train.classes[i] for i, _ in ordered]
values = [v for _, v in ordered]

plt.figure(figsize=(16, 5))
plt.bar(names, values)
plt.xticks(rotation=90, fontsize=8)
plt.ylabel("images")
plt.title("PlantVillage train class distribution (n = %d)" % len(pv_train))
plt.show()

## Where things live

**Durable on Google Drive** (survives sessions - the thing to re-verify with the Step 7 gate):
```
<DATA_DIR> = /content/drive/MyDrive/folium/data
  plantvillage/raw/<class>/*.jpg    38 classes, 54,305 images
  plantdoc/raw/{train,test}/<class>/*.jpg   28 train / 27 test classes, 2,578 images
  plantvillage/manifest.json
  plantdoc/manifest.json              per-class counts measured at download time
```

**Per-session, local** (rebuilt each run - deterministic, no need to persist):
```
<LOCAL_DATA_DIR> = /content/folium_data
  plantvillage/{train,val,test}/<class>/*.jpg   80/10/10, seed 42
  plantdoc/{train,val,test}/<class>/*.jpg       shipped test + 90/10
  class_map.json                                 PlantDoc -> PlantVillage alignment
```

**Notes**
- `WORK_DIR` (`/content/folium_cache`) is session-only scratch - wiped on reset, as expected.
- Run `organize` again after any fresh download - it is idempotent (overwrites).
- The PlantDoc clone loses 6 files on case-insensitive filesystems; Colab/Linux keeps all 2,578.
- The Step 7 durability gate is what tells you the download actually committed to Drive - the
  session's own reads can lie (FUSE cache), so always re-check from a fresh session.